# 01 数据下载

**课程**：数据分析  
**姓名**：廖婉琼  
**学号**：25210178  

本 Notebook 完成：
1. 下载10只A股个股后复权日度行情（2020-01-01至今）
2. 下载沪深300和上证综指日度数据
3. 下载宏观指标（CPI、汇率、M2、LPR、工业增加值）
4. 下载财务指标（ROE、净利润率、资产负债率、营收增速）
5. 所有下载过程记录到 download_log.txt

In [ ]:
import os
import time
import datetime
import pandas as pd
import numpy as np
import akshare as ak
import warnings
warnings.filterwarnings('ignore')

# 项目根目录
BASE_DIR = os.path.abspath('..')
DATA_DIR = os.path.join(BASE_DIR, 'data')
STOCK_DIR = os.path.join(DATA_DIR, 'stock')
INDEX_DIR = os.path.join(DATA_DIR, 'index')
MACRO_DIR = os.path.join(DATA_DIR, 'macro')
FINANCE_DIR = os.path.join(DATA_DIR, 'finance')
LOG_FILE = os.path.join(BASE_DIR, 'download_log.txt')

# 时间范围
START_DATE = '20200101'
END_DATE = datetime.datetime.now().strftime('%Y%m%d')

print(f'项目目录: {BASE_DIR}')
print(f'数据时间范围: {START_DATE} ~ {END_DATE}')

## 0. 下载日志函数

In [ ]:
def log_download(name, success, shape=None, error=None):
    """
    记录每次下载结果到 download_log.txt
    
    Parameters
    ----------
    name : str - 数据名称（如 stock_002594）
    success : bool - 是否成功
    shape : tuple - 数据形状 (可选)
    error : str - 错误信息 (可选)
    """
    timestamp = datetime.datetime.now().strftime('[%Y-%m-%d %H:%M:%S]')
    if success:
        msg = f'{timestamp} SUCCESS  {name}  shape={shape}'
    else:
        msg = f'{timestamp} FAILED   {name}  Error: {error}'
    
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(msg + '\n')
    
    print(msg)
    return success

# 初始化日志文件（清空旧日志）
with open(LOG_FILE, 'w', encoding='utf-8') as f:
    f.write(f'# 数据下载日志 - 廖婉琼 25210178\n')
    f.write(f'# 开始时间: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
    f.write(f'# 数据范围: {START_DATE} ~ {END_DATE}\n')
    f.write('#' * 60 + '\n')

print('日志函数已定义，日志文件已初始化。')

## 1. 下载个股后复权日度行情

使用 akshare 的 `stock_zh_a_hist` 接口，参数 `adjust='hfq'` 获取后复权数据。
下载字段：日期、开盘价、收盘价、最高价、最低价、成交量、成交额。

In [ ]:
# 10只股票信息
stocks = [
    {'code': '002594', 'name': '比亚迪',   'industry': '汽车'},
    {'code': '601633', 'name': '长城汽车', 'industry': '汽车'},
    {'code': '600519', 'name': '贵州茅台', 'industry': '白酒'},
    {'code': '000858', 'name': '五粮液',   'industry': '白酒'},
    {'code': '601857', 'name': '中国石油', 'industry': '能源'},
    {'code': '601088', 'name': '中国神华', 'industry': '能源'},
    {'code': '600941', 'name': '中国移动', 'industry': '通讯'},
    {'code': '000063', 'name': '中兴通讯', 'industry': '通讯'},
    {'code': '002352', 'name': '顺丰控股', 'industry': '物流'},
    {'code': '600233', 'name': '圆通速递', 'industry': '物流'},
]

print(f'共 {len(stocks)} 只股票，覆盖 {len(set(s["industry"] for s in stocks))} 个行业')
for s in stocks:
    print(f'  {s["code"]} {s["name"]} ({s["industry"]})')

In [ ]:
def download_stock(code, name, start_date, end_date, retries=3):
    """
    下载单只股票的后复权日度行情
    
    Parameters
    ----------
    code : str - 股票代码（如 '002594'）
    name : str - 股票名称
    start_date : str - 起始日期 'YYYYMMDD'
    end_date : str - 结束日期 'YYYYMMDD'
    retries : int - 重试次数
    
    Returns
    -------
    pd.DataFrame or None
    """
    for attempt in range(retries):
        try:
            df = ak.stock_zh_a_hist(
                symbol=code,
                period='daily',
                start_date=start_date,
                end_date=end_date,
                adjust='hfq'  # 后复权
            )
            if df is None or df.empty:
                log_download(f'stock_{code}', False, 
                            error='No data returned')
                return None
            
            # 标准化列名
            col_map = {
                '日期': 'date',
                '开盘': 'open',
                '收盘': 'close',
                '最高': 'high',
                '最低': 'low',
                '成交量': 'volume',
                '成交额': 'amount'
            }
            df = df.rename(columns=col_map)
            
            # 只保留需要的列
            keep_cols = ['date', 'open', 'close', 'high', 'low', 'volume', 'amount']
            df = df[[c for c in keep_cols if c in df.columns]]
            
            # 添加股票代码和名称
            df.insert(0, 'code', code)
            df.insert(1, 'name', name)
            
            log_download(f'stock_{code}_{name}', True, shape=df.shape)
            return df
            
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2)
                continue
            log_download(f'stock_{code}_{name}', False, error=str(e))
            return None

# 下载所有股票
stock_data = {}
for s in stocks:
    print(f'\n正在下载 {s["code"]} {s["name"]}...')
    df = download_stock(s['code'], s['name'], START_DATE, END_DATE)
    if df is not None:
        stock_data[s['code']] = df
        # 保存为CSV
        filepath = os.path.join(STOCK_DIR, f'stock_{s["code"]}.csv')
        df.to_csv(filepath, index=False, encoding='utf-8-sig')
        print(f'  已保存: {filepath}')
    time.sleep(1)  # 礼貌性延迟，避免被限流

print(f'\n个股下载完成: 成功 {len(stock_data)}/{len(stocks)} 只')

## 2. 下载市场指数日度数据

- **沪深300（000300）**：必选，CAPM模型市场基准
- **上证综指（000001）**：自选，反映上海交易所整体走势

In [ ]:
def download_index(code, name, start_date, end_date, retries=3):
    """
    下载指数日度数据
    """
    for attempt in range(retries):
        try:
            df = ak.stock_zh_index_daily_em(symbol=f'sh{code}')
            if df is None or df.empty:
                log_download(f'index_{code}', False, error='No data returned')
                return None
            
            # 标准化列名
            col_map = {
                'date': 'date',
                'open': 'open',
                'close': 'close',
                'high': 'high',
                'low': 'low',
                'volume': 'volume'
            }
            df.columns = [c.lower() for c in df.columns]
            
            # 筛选时间范围
            df['date'] = pd.to_datetime(df['date'])
            start_dt = pd.to_datetime(start_date)
            end_dt = pd.to_datetime(end_date)
            df = df[(df['date'] >= start_dt) & (df['date'] <= end_dt)].copy()
            
            if df.empty:
                log_download(f'index_{code}', False, error='No data in date range')
                return None
            
            # 添加指数代码
            df.insert(0, 'code', code)
            df.insert(1, 'name', name)
            
            log_download(f'index_{code}_{name}', True, shape=df.shape)
            return df
            
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2)
                continue
            log_download(f'index_{code}', False, error=str(e))
            return None

# 下载指数
indices = [
    {'code': '000300', 'name': '沪深300'},
    {'code': '000001', 'name': '上证综指'},
]

index_data = {}
for idx in indices:
    print(f'\n正在下载指数 {idx["code"]} {idx["name"]}...')
    df = download_index(idx['code'], idx['name'], START_DATE, END_DATE)
    if df is not None:
        index_data[idx['code']] = df
        filepath = os.path.join(INDEX_DIR, f'index_{idx["code"]}.csv')
        df.to_csv(filepath, index=False, encoding='utf-8-sig')
        print(f'  已保存: {filepath}')
    time.sleep(1)

print(f'\n指数下载完成: 成功 {len(index_data)}/{len(indices)} 个')

## 3. 下载宏观经济指标（月度）

下载5项宏观指标：
1. CPI同比增速（必选）
2. 人民币/美元汇率
3. M2同比增速
4. 1年期LPR利率
5. 工业增加值增速

In [ ]:
### 3.1 CPI 同比增速（月度）
print('=' * 60)
print('下载 CPI 同比增速...')
print('=' * 60)

try:
    # 使用 akshare 获取 CPI 月度数据
    cpi_df = ak.macro_china_cpi_yearly()
    print(f'原始数据形状: {cpi_df.shape}')
    print(cpi_df.head())
    print(f'列名: {cpi_df.columns.tolist()}')
except Exception as e:
    print(f'macro_china_cpi_yearly 失败: {e}')
    
# 尝试其他 CPI 接口
try:
    cpi_df2 = ak.macro_china_cpi_monthly()
    print(f'\nmacro_china_cpi_monthly 数据形状: {cpi_df2.shape}')
    print(cpi_df2.head())
    print(f'列名: {cpi_df2.columns.tolist()}')
except Exception as e:
    print(f'macro_china_cpi_monthly 也失败: {e}')

In [ ]:
### 3.2 人民币/美元汇率（月度）
print('=' * 60)
print('下载 人民币/美元汇率...')
print('=' * 60)

try:
    fx_df = ak.macro_china_exchange_rate()
    print(f'数据形状: {fx_df.shape}')
    print(fx_df.head())
    print(f'列名: {fx_df.columns.tolist()}')
except Exception as e:
    print(f'macro_china_exchange_rate 失败: {e}')

In [ ]:
### 3.3 M2 同比增速（月度）
print('=' * 60)
print('下载 M2 同比增速...')
print('=' * 60)

try:
    m2_df = ak.macro_china_money_supply()
    print(f'数据形状: {m2_df.shape}')
    print(m2_df.head())
    print(f'列名: {m2_df.columns.tolist()}')
except Exception as e:
    print(f'macro_china_money_supply 失败: {e}')

In [ ]:
### 3.4 LPR 利率
print('=' * 60)
print('下载 LPR 利率...')
print('=' * 60)

try:
    lpr_df = ak.macro_china_lpr()
    print(f'数据形状: {lpr_df.shape}')
    print(lpr_df.head())
    print(f'列名: {lpr_df.columns.tolist()}')
except Exception as e:
    print(f'macro_china_lpr 失败: {e}')

In [ ]:
### 3.5 工业增加值增速
print('=' * 60)
print('下载 工业增加值增速...')
print('=' * 60)

try:
    ind_df = ak.macro_china_industrial_production()
    print(f'数据形状: {ind_df.shape}')
    print(ind_df.head())
    print(f'列名: {ind_df.columns.tolist()}')
except Exception as e:
    print(f'macro_china_industrial_production 失败: {e}')

### 3.6 宏观数据标准化与保存

先运行上面的探索性单元格，确认每个接口的数据格式，然后在这里进行标准化处理和保存。
（实际列名和处理逻辑需根据上面的输出调整）

In [ ]:
# ============================================================
# 宏观数据标准化与保存
# 注意：以下处理逻辑需根据上面探索单元格的实际输出进行调整
# ============================================================

def standardize_and_save_macro(raw_df, indicator_name, date_col, value_col, 
                               filename, date_format=None):
    """
    将原始宏观数据标准化为统一格式并保存
    
    标准格式: date, indicator, value
    """
    df = raw_df.copy()
    
    # 确保日期列存在
    if date_col not in df.columns:
        # 尝试找到日期列
        for c in df.columns:
            if '月' in str(c) or '日' in str(c) or 'date' in str(c).lower() or '时间' in str(c):
                date_col = c
                break
    
    # 转换日期
    df[date_col] = pd.to_datetime(df[date_col], format=date_format, errors='coerce')
    
    # 筛选时间范围（2020年至今）
    df = df[df[date_col] >= '2020-01-01'].copy()
    
    # 重命名列
    df = df.rename(columns={date_col: 'date', value_col: 'value'})
    df = df[['date', 'value']].copy()
    df['indicator'] = indicator_name
    df = df[['date', 'indicator', 'value']]
    
    # 保存
    filepath = os.path.join(MACRO_DIR, f'macro_{filename}.csv')
    df.to_csv(filepath, index=False, encoding='utf-8-sig')
    log_download(f'macro_{filename}', True, shape=df.shape)
    print(f'  已保存: {filepath} ({df.shape[0]} 行)')
    return df

# 以下需根据上面的实际数据格式调整参数
macro_data = {}
print('请先运行上面的探索单元格确认数据格式，再调整本单元格的参数后运行。')

## 4. 下载财务指标

获取10只股票最近5个年度的财务指标：
- ROE（净资产收益率）
- 净利润率
- 资产负债率
- 营业收入增速

整理为长格式（Long format）：`code, year, indicator, value`

In [ ]:
# 先探索 akshare 财务数据接口
print('探索 akshare 财务数据接口...')
print()

# 尝试 stock_financial_analysis_indicator
try:
    sample = ak.stock_financial_analysis_indicator(symbol='600519')
    print(f'stock_financial_analysis_indicator (600519 贵州茅台):')
    print(f'  形状: {sample.shape}')
    print(f'  列名: {sample.columns.tolist()}')
    print(sample.head())
except Exception as e:
    print(f'stock_financial_analysis_indicator 失败: {e}')

In [ ]:
def download_financial_data(code, name, retries=3):
    """
    下载单只股票的财务指标
    返回长格式 DataFrame: code, year, indicator, value
    """
    for attempt in range(retries):
        try:
            df = ak.stock_financial_analysis_indicator(symbol=code)
            if df is None or df.empty:
                log_download(f'finance_{code}', False, error='No data')
                return None
            
            # 打印列名供参考
            if attempt == 0:
                print(f'  {code} 原始列名: {df.columns.tolist()[:10]}...')
            
            log_download(f'finance_{code}_{name}', True, shape=df.shape)
            return df
            
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2)
                continue
            log_download(f'finance_{code}_{name}', False, error=str(e))
            return None

# 下载财务数据
finance_raw = {}
for s in stocks:
    print(f'\n正在下载 {s["code"]} {s["name"]} 财务数据...')
    df = download_financial_data(s['code'], s['name'])
    if df is not None:
        finance_raw[s['code']] = df
    time.sleep(1)

print(f'\n财务数据下载完成: 成功 {len(finance_raw)}/{len(stocks)} 只')

In [ ]:
### 财务数据标准化为长格式
### 注意：以下字段名需根据上面的实际输出调整

def standardize_finance(raw_df, code, name):
    """
    将原始财务数据标准化为长格式
    输出: code, year, indicator, value
    """
    df = raw_df.copy()
    
    # TODO: 根据实际列名调整以下映射
    # 常见 akshare 财务列名示例（需根据实际情况修改）
    indicator_map = {
        '净资产收益率(%)': 'ROE',
        '销售净利率(%)': '净利润率',
        '资产负债率(%)': '资产负债率',
        '营业收入同比增长率(%)': '营业收入增速',
    }
    
    # 识别年份列
    year_col = None
    for c in df.columns:
        if '日期' in str(c) or '截止' in str(c) or '报告期' in str(c):
            year_col = c
            break
    
    if year_col is None:
        print(f'  警告: 未找到日期列，使用第一列')
        year_col = df.columns[0]
    
    # 提取年份
    df[year_col] = pd.to_datetime(df[year_col], errors='coerce')
    df['year'] = df[year_col].dt.year
    
    # 只保留近5年
    current_year = datetime.datetime.now().year
    df = df[df['year'] >= current_year - 5]
    
    # 找到存在的指标列
    records = []
    for orig_col, indicator_name in indicator_map.items():
        # 模糊匹配列名
        matched_col = None
        for c in df.columns:
            if orig_col.replace('%', '').replace('(', '').replace(')', '') in c.replace('%', '').replace('(', '').replace(')', ''):
                matched_col = c
                break
        if matched_col and matched_col in df.columns:
            for _, row in df[['year', matched_col]].iterrows():
                records.append({
                    'code': code,
                    'name': name,
                    'year': row['year'],
                    'indicator': indicator_name,
                    'value': pd.to_numeric(row[matched_col], errors='coerce')
                })
    
    return pd.DataFrame(records)

# 合并所有股票的财务数据
finance_records = []
for s in stocks:
    if s['code'] in finance_raw:
        df_long = standardize_finance(finance_raw[s['code']], s['code'], s['name'])
        if df_long is not None and not df_long.empty:
            finance_records.append(df_long)
            print(f'  {s["code"]} {s["name"]}: {len(df_long)} 条记录')

if finance_records:
    finance_all = pd.concat(finance_records, ignore_index=True)
    # 保存为长格式 CSV
    filepath = os.path.join(FINANCE_DIR, 'finance_ratios.csv')
    finance_all.to_csv(filepath, index=False, encoding='utf-8-sig')
    log_download('finance_ratios_all', True, shape=finance_all.shape)
    print(f'\n财务数据已保存: {filepath} ({finance_all.shape[0]} 行)')
    print(finance_all.head(10))
else:
    print('未获取到财务数据，请检查上面的接口返回结果。')

## 5. 下载汇总

In [ ]:
print('=' * 60)
print('下载汇总')
print('=' * 60)

# 统计文件
sections = {
    'stock': STOCK_DIR,
    'index': INDEX_DIR,
    'macro': MACRO_DIR,
    'finance': FINANCE_DIR,
}

for name, d in sections.items():
    files = [f for f in os.listdir(d) if f.endswith('.csv')]
    print(f'\n{name.upper()}/ ({len(files)} 个文件):')
    for f in sorted(files):
        fpath = os.path.join(d, f)
        size_kb = os.path.getsize(fpath) / 1024
        # 读取行列数
        try:
            tmp = pd.read_csv(fpath, nrows=0)
            tmp_full = pd.read_csv(fpath)
            print(f'  {f}: {tmp_full.shape[0]} 行 x {tmp_full.shape[1]} 列, {size_kb:.1f} KB')
        except:
            print(f'  {f}: {size_kb:.1f} KB')

print(f'\n日志文件: {LOG_FILE}')
with open(LOG_FILE, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    success_count = sum(1 for l in lines if 'SUCCESS' in l)
    failed_count = sum(1 for l in lines if 'FAILED' in l)
    print(f'  总计: {success_count} 成功, {failed_count} 失败')

## 数据下载完成！

请检查：
1. `data/stock/` 下是否有 10 个 CSV 文件
2. `data/index/` 下是否有 2 个 CSV 文件
3. `data/macro/` 下是否有 5 个 CSV 文件
4. `data/finance/` 下是否有 1 个长格式 CSV
5. `download_log.txt` 是否记录了所有下载过程

如有 FAILED 记录，请检查网络连接后重新运行对应单元格。